<a href="https://colab.research.google.com/github/maclandrol/cours-ia-med/blob/master/devoirs/Homework_Chest_XRay_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Devoir 2: Estimation d'Âge sur Radiographies Thoraciques

**Enseignant:** Emmanuel Noutahi, PhD

**Note:** /20 points

---

## Objectif

Utiliser un modèle d'IA pré-entraîné pour estimer l'âge de patients à partir de radiographies thoraciques et analyser les performances.

## Contexte Clinique

Les radiographies thoraciques contiennent des indices sur l'âge du patient:
- Calcifications vasculaires et valvulaires
- Ostéoporose et densité osseuse
- Modifications de la silhouette cardiaque
- Changements pulmonaires liés à l'âge

**Applications possibles**:
- Vérification d'identité en urgence
- Détection d'erreurs dans les dossiers
- Estimation d'âge en médecine légale

## Barème (/20)

1. **Setup et Chargement (4 pts)**: Modèle et données
2. **Analyse et Prédictions (8 pts)**: Tests et visualisations
3. **Évaluation (5 pts)**: Métriques et erreurs
4. **Réflexion (3 pts)**: Applications et limites

---

## Ressources

- **Documentation**: https://mlmed.org/torchxrayvision/
- **Paper**: Ieki et al. 2022 - *Deep learning-based age estimation from chest X-rays*
- **Dataset**: NIH Chest X-ray 14 (https://huggingface.co/datasets/BahaaEldin0/NIH-Chest-Xray-14)

## Partie 1: Setup et Chargement (4 points)

In [ ]:
!pip install -q torchxrayvision torch torchvision datasets matplotlib pandas scikit-learn seaborn pillow

In [ ]:
import torchxrayvision as xrv
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import mean_absolute_error, r2_score
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

### Question 1.1 (2 pts): Charger le modèle d'âge

Le modèle **Riken AgeModel** est entraîné sur le dataset NIH et prédit l'âge avec une MAE de 3.67 ans.

**Votre tâche**: Chargez le modèle `xrv.baseline_models.riken.AgeModel()`

In [ ]:
#===========================
# TODO: Chargez le modèle AgeModel
# 1. Créer le modèle: model = xrv.baseline_models.riken.AgeModel()
# 2. Déplacer sur device: model.to(device)
# 3. Mode évaluation: model.eval()

model = # Complétez
model = model.to(device)
model.eval()

#===========================
print(f"Modèle chargé!")
print(f"Cible: {model.targets}")

### Question 1.2 (2 pts): Charger le dataset NIH Chest X-ray

Nous utilisons le dataset NIH Chest X-ray 14 disponible sur HuggingFace.

**Votre tâche**: Chargez le split `test` et sous-échantillonnez 1000 images avec `seed=42`.

In [ ]:
# Chargement du dataset (peut prendre quelques minutes)
print("Chargement du dataset NIH Chest X-ray 14...")
dataset = load_dataset("BahaaEldin0/NIH-Chest-Xray-14", split="test")
print(f"Dataset chargé: {len(dataset)} images")

#===========================
# TODO: Sous-échantillonnez 1000 images avec seed=42
# Indice: dataset.shuffle(seed=42).select(range(1000))

dataset_sample = # Complétez

#===========================
print(f"\nÉchantillon: {len(dataset_sample)} images")
print(f"\nColonnes disponibles: {dataset_sample.column_names}")
print(f"\nPremier exemple (examinez la structure pour trouver les noms de colonnes):") 
for key, value in dataset_sample[0].items():
    if key == 'image':
        print(f"  {key}: <Image PIL>")
    else:
        print(f"  {key}: {value}")

### Question 1.3 (Bonus): Explorer le dataset

Affichez quelques statistiques sur les âges.

In [ ]:
#===========================
# TODO: Extraire les âges et afficher des statistiques
# 1. Trouvez le nom de la colonne d'âge en examinant dataset_sample[0] ci-dessus
#    (Probablement 'Patient Age' ou 'age' ou 'patient_age')
# 2. Extrayez les âges: ages = [item['NOM_COLONNE'] for item in dataset_sample]
# 3. Le reste du code fonctionne automatiquement

ages = [item['TODO_REMPLACEZ_PAR_NOM_COLONNE'] for item in dataset_sample]

#===========================
print("\nStatistiques d'âge:")
print(pd.Series(ages).describe())

# Histogramme
plt.figure(figsize=(10, 5))
plt.hist(ages, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
plt.xlabel('Âge (années)', fontsize=12)
plt.ylabel('Nombre de patients', fontsize=12)
plt.title('Distribution des Âges dans le Dataset', fontsize=13, fontweight='bold')
plt.grid(alpha=0.3)
plt.show()"

## Partie 2: Analyse et Prédictions (8 points)

### Question 2.1 (3 pts): Fonction de prédiction

Créez une fonction qui prédit l'âge d'une radiographie.

In [ ]:
def predict_age(image, model):
    """
    Prédit l'âge à partir d'une image.
    
    Args:
        image: Image PIL ou array numpy
        model: Modèle AgeModel
    
    Returns:
        float: Âge prédit en années
    """
    with torch.no_grad():
        #===========================
        # TODO: Préparez l'image
        # Option 1 (RECOMMANDÉ): Utiliser xrv.utils.load_image() 
        #   - Mais il faut d'abord sauvegarder l'image PIL en fichier temporaire
        # Option 2 (PLUS SIMPLE): Normaliser manuellement
        #   - Convertir PIL en numpy: img_np = np.array(image)
        #   - Normaliser: img_norm = xrv.datasets.normalize(img_np, 255)
        #   - Grayscale si besoin: if len(img_norm.shape) > 2: img_norm = img_norm[:,:,0]
        #   - Ajouter dimension canal: img_norm = img_norm[None, :, :]
        
        # Votre code de préparation ici (choisissez Option 1 OU 2)
        
        
        #===========================
        # TODO: Convertir en tensor et prédire
        # 1. img_tensor = torch.from_numpy(img_prepared)
        # 2. img_tensor = img_tensor.to(device)
        # 3. img_tensor = img_tensor.unsqueeze(0)  # Batch dimension
        # 4. pred = model(img_tensor)
        # 5. age_pred = float(pred[0][0].cpu())
        
        # Votre code de prédiction ici
        
        
        #===========================
    
    return age_pred"

### Question 2.2 (2 pts): Tester sur un exemple

Testez votre fonction sur la première image du dataset.

In [ ]:
#===========================
# TODO: Testez sur la première image
# 1. Récupérer l'image: dataset_sample[0]['image'] (ou le bon nom de colonne)
# 2. Récupérer l'âge réel
# 3. Prédire l'âge
# 4. Calculer l'erreur

example_image = # Complétez
true_age = # Complétez
predicted_age = # Complétez
error = # Complétez

#===========================
# Affichage
plt.figure(figsize=(6, 6))
plt.imshow(example_image, cmap='gray')
plt.title(f"Âge réel: {true_age} ans\nÂge prédit: {predicted_age:.1f} ans\nErreur: {error:.1f} ans")
plt.axis('off')
plt.show()

### Question 2.3 (3 pts): Analyser un sous-ensemble

Analysez les 100 premières images pour obtenir un aperçu des performances.

In [ ]:
#===========================
# TODO: Boucle sur les 100 premières images
# 1. Pour chaque image, prédire l'âge
# 2. Stocker dans une liste: {'age_reel': ..., 'age_predit': ..., 'erreur': ...}

results = []
n_samples = 100

print(f"Analyse de {n_samples} images...")
for i in range(n_samples):
    print(f"\rImage {i+1}/{n_samples}", end="")
    
    # Votre code ici
    
print("\nTerminé!")

results_df = pd.DataFrame(results)

#===========================
print("\nRésultats:")
print(results_df.head(10))

## Partie 3: Évaluation (5 points)

### Question 3.1 (2 pts): Métriques de performance

In [ ]:
#===========================
# TODO: Calculez les métriques
# 1. MAE (Mean Absolute Error)
# 2. Erreur médiane
# 3. R² score
# 4. Pourcentage d'erreurs < 5 ans

mae = # Complétez
median_error = # Complétez
r2 = # Complétez
percent_within_5 = # Complétez

#===========================
print("\nMÉTRIQUES DE PERFORMANCE")
print("="*50)
print(f"MAE: {mae:.2f} ans")
print(f"Erreur médiane: {median_error:.2f} ans")
print(f"R² score: {r2:.3f}")
print(f"Prédictions à ±5 ans: {percent_within_5:.1f}%")
print("="*50)
print(f"\nRappel: Le paper original rapporte MAE = 3.67 ans")

### Question 3.2 (2 pts): Visualisations

In [ ]:
#===========================
# TODO: Créez 2 graphiques
# 1. Scatter plot: Âge réel vs Âge prédit (avec ligne y=x)
# 2. Histogramme des erreurs

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1: Scatter plot
# Votre code ici

# Graphique 2: Histogramme des erreurs
# Votre code ici

#===========================
plt.tight_layout()
plt.show()

### Question 3.3 (1 pt): Analyse par groupe d'âge

Le modèle performe-t-il mieux sur certaines tranches d'âge?

In [ ]:
#===========================
# TODO: Créez des groupes d'âge et calculez la MAE par groupe
# Groupes suggérés: <30, 30-50, 50-70, >70

def categorize_age(age):
    # Complétez cette fonction
    pass

results_df['groupe_age'] = # Complétez

# Calculez la MAE par groupe
# Votre code ici

#===========================

## Partie 4: Réflexion (3 points)

### Question 4.1 (1 pt): Applications cliniques

Rédigez 5-7 lignes expliquant:
1. Dans quels cas cliniques l'estimation d'âge par radiographie serait utile?
2. Les performances observées (MAE ~3-4 ans) sont-elles suffisantes pour ces usages?

**Votre réponse:**

[Écrivez ici]

### Question 4.2 (1 pt): Limites

Listez 3 limites importantes de ce modèle.

**Vos limites:**

1. [Votre limite]
2. [Votre limite]
3. [Votre limite]

### Question 4.3 (1 pt): Éthique et biais

Le modèle a été entraîné principalement sur des patients américains.

**Question**: Quels biais potentiels cela pourrait-il introduire? (3-5 lignes)

**Votre réponse:**

[Écrivez ici]

## BONUS: Testez avec votre propre image!

Uploadez une radiographie thoracique et testez le modèle.

In [ ]:
from google.colab import files
import io

# Upload
print("Uploadez une radiographie thoracique:")
uploaded = files.upload()

if uploaded:
    # Charger la première image uploadée
    filename = list(uploaded.keys())[0]
    image = Image.open(io.BytesIO(uploaded[filename]))
    
    # Prédire
    predicted_age = predict_age(image, model)
    
    # Afficher
    plt.figure(figsize=(8, 8))
    plt.imshow(image, cmap='gray')
    plt.title(f"Âge prédit: {predicted_age:.1f} ans", fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.show()
    
    print(f"\nÂge estimé: {predicted_age:.1f} ans")
    print(f"Intervalle de confiance approximatif: [{predicted_age-3.67:.1f}, {predicted_age+3.67:.1f}] ans")
else:
    print("Aucune image uploadée.")

## Remise

**Avant de soumettre:**
1. Exécutez: Exécution → Tout exécuter
2. Vérifiez qu'il n'y a pas d'erreurs
3. Vérifiez que toutes les réponses textuelles sont complètes
4. Sauvegardez une copie dans votre Drive
5. Partagez le lien Colab

**Bon travail!**